In [ ]:
import os
import time
import argparse
import json
from pathlib import Path
from typing import List, Optional, Tuple, Literal

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset, ConcatDataset
from scipy.ndimage import zoom
from sklearn.metrics import classification_report, confusion_matrix
from torchvision.models import (
    resnet18, ResNet18_Weights,
    efficientnet_b0, EfficientNet_B0_Weights,
)

# ─── Constants ───────────────────────────────────────────────────────────────
MICRO_WINDOW    = 64        # 64-min context for 15-min ahead prediction
STRUCT_WINDOW   = 240       # 4 hours of context
PAA_BINS        = 60        # PAA output bins for structural resolution
IMAGE_SIZE      = 64        # final CNN image size (H = W)
LABEL_HORIZON   = 15        # 15 minutes ahead (intraday)
LABEL_THRESHOLD = 0.001      # 0.1% threshold for 15-min signals (was 0.05%)
CLASS_NAMES     = {0: "Short", 1: "Long"}


In [ ]:
def load_ohlcv(path: str) -> pd.DataFrame:
    """Load a single symbol's 1-min OHLCV CSV, handles corrupt first rows and headers."""
    if not os.path.exists(path):
        raise FileNotFoundError(f"No data file at '{path}'")
    
    expected_cols = ['date', 'open', 'high', 'low', 'close', 'volume']
    
    # Read the file and filter for rows that look like data
    valid_lines = []
    with open(path, 'r', encoding='utf-8', errors='ignore') as f:
        for line in f:
            parts = line.strip().split(',')
            if len(parts) in [5, 6]:
                # Skip if it's a header or contains non-numeric price data in col 1
                if "date" in parts[0].lower() or parts[1].strip().isalpha():
                    continue
                valid_lines.append(line)
    
    if not valid_lines:
        raise ValueError(f"No valid data rows found in {path}")

    # Create DataFrame from valid lines, explicitly avoiding mixed dtype warnings
    from io import StringIO
    df = pd.read_csv(StringIO("".join(valid_lines)), header=None, low_memory=False)
    
    # Assign column names based on count
    if len(df.columns) == 6:
        df.columns = expected_cols
    elif len(df.columns) == 5:
        df.columns = expected_cols[:5]
        df['volume'] = 0.0 # Fill missing volume
    
    # Robust date parsing
    try:
        # Try to parse with standard ISO format first for speed
        df["date"] = pd.to_datetime(df[df.columns[0]], utc=True, errors='coerce').dt.tz_localize(None)
    except:
        # Fallback to slower inference if needed
        df["date"] = pd.to_datetime(df[df.columns[0]], errors='coerce').dt.tz_localize(None)
    
    # Drop rows where date or close price failed to parse
    df = df.dropna(subset=['date', 'close'])
    
    # Ensure numeric columns are actually numeric
    for col in ['open', 'high', 'low', 'close', 'volume']:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    
    df = df.sort_values("date").reset_index(drop=True)
    return df

def paa(series: np.ndarray, n_bins: int) -> np.ndarray:
    n = len(series)
    if n == n_bins: return series.copy()
    if n % n_bins == 0: return series.reshape(n_bins, n // n_bins).mean(axis=1)
    cs  = np.concatenate([[0.0], np.cumsum(series)])
    idx = np.linspace(0, n, n_bins + 1)
    lo, hi = np.floor(idx).astype(int), np.ceil(idx).astype(int)
    frac = idx - lo
    hi   = np.minimum(hi, n)
    interp = cs[lo] + frac * (cs[hi] - cs[lo])
    return np.diff(interp) / np.diff(idx)

def zscore(series: np.ndarray, eps: float = 1e-8) -> np.ndarray:
    z = (series - series.mean()) / (series.std() + eps)
    return np.clip(z, -3.0, 3.0)

class RGBMarketFingerprint:
    def __init__(self, image_size: int = IMAGE_SIZE, n_bins: int = 8, max_n: int = 1000):
        self.image_size = image_size
        self.n_bins     = n_bins
        self.max_n      = max_n

    @staticmethod
    def _minmax(arr: np.ndarray) -> np.ndarray:
        lo, hi = arr.min(), arr.max()
        span = hi - lo
        if span < 1e-9: return np.zeros_like(arr)
        return (arr - lo) / span

    @staticmethod
    def _rescale_to_minus1_plus1(series: np.ndarray) -> np.ndarray:
        lo, hi = series.min(), series.max()
        span = hi - lo
        if span < 1e-9: return np.zeros_like(series)
        return 2.0 * (series - lo) / span - 1.0

    def _gasf(self, x: np.ndarray) -> np.ndarray:
        """Optimized GASF using trig identities: cos(A+B) = cosA*cosB - sinA*sinB"""
        x = np.clip(x, -1.0, 1.0)
        sin_phi = np.sqrt(np.clip(1.0 - x**2, 0, 1))
        return np.outer(x, x) - np.outer(sin_phi, sin_phi)

    def _gadf(self, x: np.ndarray) -> np.ndarray:
        """Optimized GADF using trig identities: sin(A-B) = sinA*cosB - cosA*sinB"""
        x = np.clip(x, -1.0, 1.0)
        sin_phi = np.sqrt(np.clip(1.0 - x**2, 0, 1))
        return np.outer(sin_phi, x) - np.outer(x, sin_phi)

    def _mtf(self, series: np.ndarray) -> np.ndarray:
        n = len(series)
        k = self.n_bins
        
        # 1. Handle bin collapse: Use unique quantiles
        quantiles = np.percentile(series, np.linspace(0, 100, k + 1))
        unique_q = np.unique(quantiles)
        
        # Fallback for constant or near-constant series
        if len(unique_q) < 2:
            return np.zeros((n, n))
            
        actual_k = len(unique_q) - 1
        
        # 2. Assign bins using side='left' (consistent with digitize)
        bins = np.digitize(series, unique_q[1:-1])
        bins = np.clip(bins, 0, actual_k - 1)
        
        # 3. Transition matrix Q
        Q = np.zeros((actual_k, actual_k), dtype=np.float64)
        np.add.at(Q, (bins[:-1], bins[1:]), 1.0)
        
        # 4. Normalize rows. Handle 'dead ends' (states only visited at the end)
        row_sums = Q.sum(axis=1, keepdims=True)
        zero_rows = (row_sums.flatten() == 0)
        if np.any(zero_rows):
            for idx in np.where(zero_rows)[0]:
                Q[idx, idx] = 1.0
            row_sums[zero_rows] = 1.0
            
        W = Q / row_sums
        
        # 5. Map back to time-domain matrix
        return W[bins[:, None], bins[None, :]]

    def _resize(self, mat: np.ndarray, is_mtf: bool = False) -> np.ndarray:
        n = mat.shape[0]
        if n == self.image_size: return mat
        # MTF uses order=0 (nearest) to preserve Markov block structure
        # GASF/GADF use order=1 (bilinear) for smoothness
        return zoom(mat, self.image_size / n, order=0 if is_mtf else 1)

    def encode(self, price: np.ndarray, volume: np.ndarray) -> np.ndarray:
        # Memory Guard
        if len(price) > self.max_n:
            price = price[-self.max_n:]
            volume = volume[-self.max_n:]
            
        # Price matrices (R and G)
        xp = self._rescale_to_minus1_plus1(price)
        gasf_p = self._gasf(xp)
        gadf_p = self._gadf(xp)
        
        # Volume matrix (B) - Using MTF to capture volume regime transitions
        mtf_v  = self._mtf(volume)
        
        # Resize and combine
        rgb = np.stack([
            self._minmax(self._resize(gasf_p, is_mtf=False)),
            self._minmax(self._resize(gadf_p, is_mtf=False)),
            self._minmax(self._resize(mtf_v,  is_mtf=True)),
        ], axis=0)
        
        return rgb.astype(np.float32)

In [ ]:
def compute_labels(close: np.ndarray, horizon: int = LABEL_HORIZON, threshold: float = LABEL_THRESHOLD) -> np.ndarray:
    n       = len(close)
    current = close[: n - horizon]
    future  = close[horizon:]
    pct     = (future - current) / (current + 1e-8)
    labels = np.zeros(len(pct), dtype=np.int64)
    labels[pct >  threshold] =  1
    labels[pct < -threshold] = -1
    return labels

class MarketFingerprintDataset(Dataset):
    def __init__(self, close, volume, labels, indices, resolution="micro", image_size=IMAGE_SIZE, transform=None):
        self.close      = close.astype(np.float64)
        self.volume     = volume.astype(np.float64)
        self.labels     = labels
        self.indices    = indices
        self.resolution = resolution
        self.transform  = transform
        self.window     = MICRO_WINDOW if resolution == "micro" else STRUCT_WINDOW
        self.encoder    = RGBMarketFingerprint(image_size=image_size)

    def __len__(self) -> int: return len(self.indices)

    def __getitem__(self, idx: int):
        start = int(self.indices[idx])
        
        # 1. Price Context (Rolling Z-Score)
        p_lookback = self.close[max(0, start - 200) : start]
        p_mean, p_std = p_lookback.mean(), p_lookback.std()
        p_window = (self.close[start : start + self.window] - p_mean) / (p_std + 1e-8)
        p_window = np.clip(p_window, -3.0, 3.0)
        
        # 2. Volume Context (Log-transformed Z-Score)
        v_lookback = np.log1p(self.volume[max(0, start - 200) : start])
        v_mean, v_std = v_lookback.mean(), v_lookback.std()
        v_window = (np.log1p(self.volume[start : start + self.window]) - v_mean) / (v_std + 1e-8)
        v_window = np.clip(v_window, -3.0, 3.0)
        
        if self.resolution == "structural":
            p_window = paa(p_window, PAA_BINS)
            v_window = paa(v_window, PAA_BINS)
            
        image = self.encoder.encode(p_window, v_window)
        label = int((self.labels[start + self.window - 1] + 1) // 2)
        img_tensor = torch.from_numpy(image)
        if self.transform: img_tensor = self.transform(img_tensor)
        return img_tensor, torch.tensor(label, dtype=torch.long)

In [ ]:
def _time_stratified_sample(indices, max_samples, n_strata=20, seed=42):
    """Sample indices while preserving coverage across time.

    Divides the timeline into n_strata equal chunks and samples
    proportionally from each. This guarantees the model sees data
    from bull, bear, sideways, and crash regimes alike.
    """
    if len(indices) <= max_samples:
        return indices

    rng = np.random.default_rng(seed)
    chunk_size = len(indices) // n_strata
    samples_per_chunk = max(1, max_samples // n_strata)

    sampled = []
    for i in range(n_strata):
        start = i * chunk_size
        end = (i + 1) * chunk_size if i < n_strata - 1 else len(indices)
        chunk = indices[start:end]
        if len(chunk) <= samples_per_chunk:
            sampled.extend(chunk)
        else:
            sampled.extend(rng.choice(chunk, samples_per_chunk, replace=False))

    return np.sort(np.array(sampled))

def build_datasets(csv_path, resolution="micro", train_ratio=0.80, val_ratio=0.10,
                   image_size=IMAGE_SIZE, max_samples_per_stock=None,
                   sample_stride=5, random_seed=42):
    window = MICRO_WINDOW if resolution == "micro" else STRUCT_WINDOW
    df     = load_ohlcv(csv_path)
    close  = df["close"].values.astype(np.float64)
    volume = df["volume"].values.astype(np.float64)
    labels = compute_labels(close)

    lookback_pad = 200
    max_idx = min(len(labels) - window + 1, len(close) - window - LABEL_HORIZON + 1)

    # Build valid indices with a stride to avoid nearly-identical consecutive windows.
    # For a 15-min horizon with 64-min context, consecutive 1-min steps share 63/64
    # of their input — training on all of them is massively redundant.
    indices = np.arange(lookback_pad, max_idx)

    # Keep only samples with a directional signal (no Hold)
    label_indices = indices + window - 1
    valid_mask = labels[label_indices] != 0
    indices = indices[valid_mask]
    if len(indices) == 0:
        raise ValueError(f"No directional samples in {csv_path} with threshold={LABEL_THRESHOLD}")

    # Apply stride to reduce redundancy
    indices = indices[::sample_stride]

    if len(indices) <= 0:
        raise ValueError(f"Not enough data in {csv_path} for window+lookback")

    # Time-stratified cap: if a stock still has too many samples after striding,
    # sample uniformly across its entire timeline so no regime is skipped.
    if max_samples_per_stock is not None and len(indices) > max_samples_per_stock:
        indices = _time_stratified_sample(
            indices, max_samples_per_stock, seed=random_seed
        )
        print(f"  [Stratified] {Path(csv_path).name}: {len(indices):,} samples "
              f"(stride={sample_stride}, capped from full history)")

    n = len(indices)
    train_idx = indices[:int(n * train_ratio)]
    val_idx   = indices[int(n * train_ratio):int(n * (train_ratio + val_ratio))]
    test_idx  = indices[int(n * (train_ratio + val_ratio)):]

    train_ds = MarketFingerprintDataset(close, volume, labels, train_idx, resolution, image_size)
    val_ds   = MarketFingerprintDataset(close, volume, labels, val_idx,   resolution, image_size)
    test_ds  = MarketFingerprintDataset(close, volume, labels, test_idx,  resolution, image_size)
    return train_ds, val_ds, test_ds

def build_multi_stock_datasets(data_path, resolution="micro", train_ratio=0.8,
                               val_ratio=0.1, image_size=IMAGE_SIZE,
                               max_samples_per_stock=20000,
                               sample_stride=5):
    path = Path(data_path)
    if path.is_file():
        print(f"Loading single symbol: {path.name}")
        return build_datasets(
            str(path), resolution, train_ratio, val_ratio, image_size,
            max_samples_per_stock=max_samples_per_stock,
            sample_stride=sample_stride
        )

    csv_files = sorted(list(path.glob("*.csv")))
    if not csv_files:
        raise FileNotFoundError(f"No CSV files found in {data_path}")

    train_datasets, val_datasets, test_datasets = [], [], []

    print(f"Loading {len(csv_files)} symbols from {data_path}...")
    for f in csv_files:
        try:
            t_ds, v_ds, te_ds = build_datasets(
                str(f), resolution, train_ratio, val_ratio, image_size,
                max_samples_per_stock=max_samples_per_stock,
                sample_stride=sample_stride
            )
            train_datasets.append(t_ds)
            val_datasets.append(v_ds)
            test_datasets.append(te_ds)
        except Exception as e:
            print(f"  [Skip] {f.name}: {e}")

    return (
        ConcatDataset(train_datasets),
        ConcatDataset(val_datasets),
        ConcatDataset(test_datasets)
    )

def compute_class_weights(dataset) -> np.ndarray:
    if isinstance(dataset, ConcatDataset):
        all_counts = np.zeros(2, dtype=np.float64)
        for ds in dataset.datasets:
            raw = ds.labels[ds.indices]
            all_counts += np.bincount(((raw + 1) // 2).astype(np.int64), minlength=2)
        counts = all_counts
    else:
        raw     = dataset.labels[dataset.indices]
        counts  = np.bincount(((raw + 1) // 2).astype(np.int64), minlength=2).astype(np.float64)

    counts  = np.maximum(counts, 1.0)
    weights = counts.sum() / (2.0 * counts)
    return weights.astype(np.float32)


In [ ]:
class MarketFingerprintCNN(nn.Module):
        num_classes: int = 2,
        super().__init__()
        if backbone == "resnet18":
            weights = ResNet18_Weights.DEFAULT if pretrained else None
            base    = resnet18(weights=weights)
            base.conv1  = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
            base.maxpool = nn.Identity()
            base.fc  = nn.Sequential(nn.Dropout(p=dropout), nn.Linear(base.fc.in_features, num_classes))
            self.model = base
        elif backbone == "efficientnet_b0":
            weights = EfficientNet_B0_Weights.DEFAULT if pretrained else None
            base    = efficientnet_b0(weights=weights)
            base.classifier = nn.Sequential(nn.Dropout(p=dropout, inplace=True), nn.Linear(base.classifier[1].in_features, num_classes))
            self.model = base
        self._softmax = nn.Softmax(dim=1)

    def forward(self, x: torch.Tensor, return_logits: bool = True) -> torch.Tensor:
        logits = self.model(x)
        return logits if return_logits else self._softmax(logits)

def build_model(backbone="resnet18", pretrained=True, dropout=0.1, device=None):
    model = MarketFingerprintCNN(backbone=backbone, pretrained=pretrained, dropout=dropout)
    if device: model = model.to(device)
    return model

In [ ]:
def train_epoch(model, loader, criterion, optimizer, device, epoch, total, scaler=None, global_step=0, save_every=500):
    model.train()
    running_loss, correct, n = 0.0, 0, 0
    t0 = time.time()
    for step, (imgs, labels) in enumerate(loader, 1):
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad(set_to_none=True)

        # Mixed precision forward + backward
        if scaler is not None:
            with torch.amp.autocast('cuda'):
                logits = model(imgs)
                loss = criterion(logits, labels)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            logits = model(imgs)
            loss = criterion(logits, labels)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

        bs = imgs.size(0)
        running_loss += loss.item() * bs
        correct += (logits.argmax(1) == labels).sum().item()
        n += bs
        global_step += 1

        # Periodic checkpoint every N steps
        if save_every > 0 and global_step % save_every == 0:
            ckpt_path = f"checkpoint_step_{global_step}.pt"
            save_checkpoint(model, ckpt_path, download=True)

        log_every = max(1, len(loader) // 10)
        if step % log_every == 0 or step == len(loader):
            print(f"  Epoch {epoch}/{total} | step {step}/{len(loader)} | loss {running_loss/n:.4f} | acc {correct/n:.4f} | {time.time()-t0:.0f}s")
    return running_loss / n, correct / n, global_step

@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    loss, correct, n = 0.0, 0, 0
    all_preds, all_labels = [], []
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        logits = model(imgs)
        loss   += criterion(logits, labels).item() * imgs.size(0)
        preds   = logits.argmax(1)
        correct += (preds == labels).sum().item()
        n       += imgs.size(0)
        all_preds.extend(preds.cpu().numpy()); all_labels.extend(labels.cpu().numpy())
    return loss/n, correct/n, np.array(all_preds), np.array(all_labels)

@torch.no_grad()
def backtest_engine(model, dataset, device, confidence_threshold=0.6, commission=0.0005):
    model.eval()

    if isinstance(dataset, ConcatDataset):
        print(f"Running multi-symbol backtest on {len(dataset.datasets)} symbols...")
        total_pnl = []
        total_trades = 0
        for ds in dataset.datasets:
            pnl_arr, trades = _run_sim(model, ds, device, confidence_threshold, commission)
            total_pnl.extend(pnl_arr)
            total_trades += trades
    else:
        pnl_arr, total_trades = _run_sim(model, dataset, device, confidence_threshold, commission)
        total_pnl = pnl_arr

    total_pnl = np.array(total_pnl)
    cum_pnl = np.cumsum(total_pnl)
    win_rate = (total_pnl > 0).sum() / total_trades if total_trades > 0 else 0

    print(f"\n{'='*40}")
    print(f"       AGGREGATE BACKTEST (Threshold: {confidence_threshold})")
    print(f"{'='*40}")
    print(f"  Total Trades:   {total_trades:,}")
    print(f"  Win Rate:       {win_rate:.2%}")
    print(f"  Final PnL:      {cum_pnl[-1]:.2%}" if len(cum_pnl)>0 else "  Final PnL: 0.0%")
    print(f"  Avg PnL/Trade:  {total_pnl[total_pnl != 0].mean():.4%}" if total_trades > 0 else "  Avg PnL/Trade: 0.0%")
    print(f"{'='*40}\n")
    return cum_pnl

def _run_sim(model, ds, device, threshold, commission):
    loader = DataLoader(ds, batch_size=128, shuffle=False)
    all_probs = []
    for imgs, _ in loader:
        logits = model(imgs.to(device))
        all_probs.append(torch.softmax(logits, dim=1).cpu().numpy())

    probs = np.concatenate(all_probs, axis=0)
    close = ds.close
    indices = ds.indices
    window = ds.window
    horizon = LABEL_HORIZON

    pnl = []
    trades = 0
    in_position = False
    position_exit_time = -1

    for i, start_idx in enumerate(indices):
        now_idx = start_idx + window - 1

        # Skip if still holding a previous position
        if in_position and now_idx < position_exit_time:
            pnl.append(0.0)
            continue

        in_position = False
        p = probs[i]

        entry_price = close[now_idx]
        exit_idx = now_idx + horizon
        exit_price = close[min(exit_idx, len(close)-1)]
        ret = (exit_price - entry_price) / entry_price

        if p[1] > threshold: # Long
            pnl.append(ret - commission); trades += 1
            in_position = True
            position_exit_time = exit_idx
        elif p[0] > threshold: # Short
            pnl.append(-ret - commission); trades += 1
            in_position = True
            position_exit_time = exit_idx
        else:
            pnl.append(0.0)
    return pnl, trades

def save_checkpoint(model, path, download=False):
    """Save model state_dict. If download=True, trigger Colab file download."""
    torch.save(model.state_dict(), path)
    print(f"  💾 Checkpoint saved: {path}")
    if download:
        try:
            from google.colab import files
            files.download(path)
            print(f"  📥 Download triggered: {path}")
        except Exception as e:
            print(f"  ⚠️ Download failed (not in Colab?): {e}")


In [ ]:
# ─── OPTIONAL: Upload data zip if running in isolated environment ────────────
# Run this only if local data is NOT found above.
# 1. Zip your data folder locally:  zip -r data.zip minute/
# 2. Run this cell, select the zip file, then run the next cell.

try:
    from ipywidgets import FileUpload
    import zipfile
    upload = FileUpload(accept='.zip', multiple=False)
    display(upload)
except ImportError:
    print("ipywidgets not installed. Use: !pip install ipywidgets")


In [ ]:
import os, subprocess, sys, shutil
from pathlib import Path
from collections import Counter

# ─── 1. DOWNLOAD DATA FROM GOOGLE DRIVE (if not present) ─────────────────────
BATCH1_DIR = "batch1"

def get_csv_count(d):
    return len([f for f in os.listdir(d) if f.endswith('.csv')]) if os.path.isdir(d) else 0

if get_csv_count(BATCH1_DIR) == 0:
    print("📥 batch1/ not found locally. Downloading from Google Drive...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "gdown", "-q"])

    FOLDER_ID = "12gpKGivt-IEYPloWHfd_-0TsJNFejPDB"
    TMP_DIR = "/tmp/gdrive_batch"
    subprocess.check_call([
        "gdown", "--folder",
        f"https://drive.google.com/drive/folders/{FOLDER_ID}",
        "-O", TMP_DIR
    ])

    # Flatten: move all CSVs from nested folders into batch1/
    os.makedirs(BATCH1_DIR, exist_ok=True)
    for root, dirs, files in os.walk(TMP_DIR):
        for f in files:
            if f.endswith('.csv'):
                src = os.path.join(root, f)
                dst = os.path.join(BATCH1_DIR, f)
                shutil.move(src, dst)
    shutil.rmtree(TMP_DIR, ignore_errors=True)
    print("✅ Download complete.")
else:
    print("✅ batch1/ already present.")

csv_files = sorted([f for f in os.listdir(BATCH1_DIR) if f.endswith('.csv')])
print(f"Found {len(csv_files)} CSV files: {csv_files[:3]}{'...' if len(csv_files)>3 else ''}")

if len(csv_files) == 0:
    raise FileNotFoundError("No CSV files found in batch1/. Check the Drive download.")

DATA_PATH = BATCH1_DIR

# ─── 2. TRAINING CONFIG (T4 Optimized, target <1 hour) ───────────────────────
BACKBONE       = "resnet18"       # pretrained features help on limited data
RESOLUTION     = "micro"
EPOCHS         = 5         # more epochs; early stopping will cut if needed
BATCH_SIZE     = 256       # T4 handles 256 easily for 64x64 images
LR             = 3e-4        # faster learning for underfitting
WORKERS        = 2         # Colab T4 VMs: 2 CPUs; persistent_workers keeps them warm
CHECKPOINT     = "best_model.pt"
BACKUP_CHECKPOINT = "checkpoint_epoch_{epoch}.pt"

# ─── 3. EXECUTION ─────────────────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if device.type != "cuda":
    print("⚠️ WARNING: CUDA not available. Training will be very slow on CPU.")

print("Building multi-stock datasets...")
train_ds, val_ds, test_ds = build_multi_stock_datasets(
    DATA_PATH, RESOLUTION,
    max_samples_per_stock=20000,   # more data per stock
    sample_stride=5                 # more diverse patterns
)
print(f"Total samples | train: {len(train_ds):,}, val: {len(val_ds):,}, test: {len(test_ds):,}")
weights = compute_class_weights(train_ds)

# ─── 4. CLASS DISTRIBUTION CHECK ─────────────────────────────────────────────
if hasattr(train_ds, 'targets'):
    dist = Counter(train_ds.targets)
elif hasattr(train_ds, 'labels'):
    dist = Counter(train_ds.labels)
else:
    all_labels = []
    for _, labels in DataLoader(train_ds, batch_size=BATCH_SIZE, num_workers=0):
        all_labels.extend(labels.numpy())
    dist = Counter(all_labels)

total = sum(dist.values())
print("\n=== CLASS DISTRIBUTION ===")
for cls, count in sorted(dist.items()):
    pct = 100 * count / total
    name = ["Short", "Long"][cls] if cls < 2 else str(cls)
    print(f"  {name}: {count:,} ({pct:.1f}%)")

if max(dist.values()) / total > 0.8:
    print("⚠️ WARNING: Severe class imbalance detected!")
    print("   Accuracy will be misleading. Use F1-macro or balanced accuracy.")

# ─── 5. DATA LOADERS & MODEL ─────────────────────────────────────────────────
loader_kw = dict(
    batch_size=BATCH_SIZE,
    num_workers=WORKERS,
    pin_memory=True if device.type=="cuda" else False,
    persistent_workers=True if WORKERS > 0 else False,
)
train_loader = DataLoader(train_ds, shuffle=True, **loader_kw)
val_loader   = DataLoader(val_ds, shuffle=False, **loader_kw)
test_loader  = DataLoader(test_ds, shuffle=False, **loader_kw)

print("Building model...")
model = build_model(BACKBONE, device=device)
criterion = nn.CrossEntropyLoss()  # remove class weights; they were biasing Buy
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=1)

# Mixed precision scaler (only on CUDA)
scaler = torch.amp.GradScaler('cuda') if device.type == 'cuda' else None

# ─── 6. TRAINING LOOP WITH EARLY STOPPING ────────────────────────────────────
best_val_loss = float("inf")
patience = 2
patience_counter = 0
global_step = 0

for epoch in range(1, EPOCHS + 1):
    t_loss, t_acc, global_step = train_epoch(
        model, train_loader, criterion, optimizer, device, epoch, EPOCHS,
        scaler=scaler, global_step=global_step, save_every=500
    )
    v_loss, v_acc, _, _ = evaluate(model, val_loader, criterion, device)

    # Step scheduler
    scheduler.step(v_loss)
    current_lr = optimizer.param_groups[0]["lr"]
    print(f'  📉 LR: {current_lr:.2e}')

    print(f"\n  ▶ Epoch {epoch} | train_loss: {t_loss:.4f} | val_loss: {v_loss:.4f} | val_acc: {v_acc:.4f}\n")

    # Save every epoch (backup)
    backup_path = BACKUP_CHECKPOINT.format(epoch=epoch)
    save_checkpoint(model, backup_path, download=False)

    # Early stopping + best model check
    if v_loss < best_val_loss:
        best_val_loss = v_loss
        save_checkpoint(model, CHECKPOINT, download=True)
        print(f"  ✔ New best model saved (val_loss: {v_loss:.4f})")
        patience_counter = 0
    else:
        patience_counter += 1
        print(f"  ⚠ No improvement ({patience_counter}/{patience})")
        if patience_counter >= patience:
            print(f"  🛑 Early stopping at epoch {epoch}")
            break

print(f"\nLoading best model from {CHECKPOINT}")
model.load_state_dict(torch.load(CHECKPOINT, map_location=device))

# ─── 7. FINAL TEST EVALUATION (Better Metrics) ───────────────────────────────
from sklearn.metrics import classification_report, f1_score, balanced_accuracy_score, confusion_matrix

_, t_acc, preds, labels = evaluate(model, test_loader, criterion, device)

print(f"\n=== FINAL TEST RESULTS ===")
print(f"Raw Accuracy:      {t_acc:.4f}")
print(f"Balanced Accuracy: {balanced_accuracy_score(labels, preds):.4f}")
print(f"F1-Macro:          {f1_score(labels, preds, average='macro'):.4f}")
print(f"F1-Weighted:       {f1_score(labels, preds, average='weighted'):.4f}")

print("\nPer-Class F1:")
print(classification_report(labels, preds, target_names=["Short", "Long"], labels=[0, 1]))

print("Confusion Matrix:")
print(confusion_matrix(labels, preds, labels=[0, 1]))

print("\nRunning Backtest Simulation on Test Set...")
backtest_engine(model, test_ds, device, confidence_threshold=0.6)
